In [1]:
import os
import sys
sys.path.append('..')
import torch
import librosa
import pandas as pd
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm
from src.modules import ResynthesisWithBottleneckAndTVTimbreModule, ResynthesisWithLogitsAndTVTimbreModule, ResynthesisWithQuantAndTVTimbreModule, SpeakerModule

2025-10-07 02:10:43 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # Set the GPU device
MAX_WAV_VALUE = 32768.0
model_name = "DarkStream-quant-TVT" # Options: DarkStream-bottleneck-TVT, DarkStream-logits-TVT, DarkStream-quant-TVT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
speaker_encoder = SpeakerModule(device="cuda")
if model_name == "DarkStream-logits-TVT":
    resynthesizer = ResynthesisWithLogitsAndTVTimbreModule.load_from_checkpoint(
        "../ckpts/resynthesis_with_logits_and_tvtimbre_epoch=16-val_loss=1.5489.ckpt",
        encoder_path=None,
        map_location="cuda"
    )
elif model_name == "DarkStream-bottleneck-TVT":
    # Uncomment the following line to use the KM Resynthesis Module instead
    resynthesizer = ResynthesisWithBottleneckAndTVTimbreModule.load_from_checkpoint(
        "../ckpts/resynthesis_bottleneck_and_tvtimbre_epoch=20-val_loss=1.7971.ckpt",
        encoder_path=None,
        decoder_pretrained=None,
        kmeans_path=None,
        map_location="cuda"
    )
elif model_name == "DarkStream-quant-TVT":
    resynthesizer = ResynthesisWithQuantAndTVTimbreModule.load_from_checkpoint(
        "../ckpts/resynthesis_quant_and_tvtimbre_epoch=17-val_loss=1.8843.ckpt",
        encoder_path=None,
        map_location="cuda"
    )

resynthesizer = resynthesizer.eval()

2025-10-07 02:10:43 | INFO | speechbrain.utils.fetching | Fetch hyperparams.yaml: Using symlink found at '/data/waris/code/DarkStreamTVT/notebooks/pretrained_models/speaker/spkrec-xvect-voxceleb/hyperparams.yaml'
2025-10-07 02:10:43 | INFO | speechbrain.utils.fetching | Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
2025-10-07 02:10:43 | DEBUG | speechbrain.utils.checkpoints | Registered checkpoint save hook for _save
2025-10-07 02:10:43 | DEBUG | speechbrain.utils.checkpoints | Registered checkpoint load hook for _load
2025-10-07 02:10:43 | DEBUG | speechbrain.utils.checkpoints | Registered parameter transfer hook for _load
/data/waris/installations/darkstream/lib/python3.10/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
2025-

In [4]:
def inference_ns(source_audio_path, tgt_embed, sample_rate=16000):
    src_audio, sr = librosa.load(source_audio_path, sr=sample_rate)
    assert sr == sample_rate
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    resynthesizer.eval()
    with torch.no_grad():
        resyth_audio = resynthesizer(src_audio, tgt_embed)
        resyth_audio = resyth_audio.squeeze()
        resyth_audio = resyth_audio * MAX_WAV_VALUE
        resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')
    return resyth_audio

In [5]:
def inference(source_audio_path, tgt_embed, model_name, use_projector=False, sample_rate=16000):
    src_audio, sr = librosa.load(source_audio_path, sr=sample_rate)
    assert sr == sample_rate
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    resynthesizer.eval()
    with torch.no_grad():
        original_length = src_audio.shape[-1]
        chunk_size = 3
        frame_samples = 320
        lookahead_size = 7 if model_name == "DarkStream-logits-TVT" else 4

        # The model advances by 'stride' frames each step.
        stride = chunk_size * frame_samples

        # We always need lookahead frames available for the last step.
        lookahead_samples = lookahead_size * frame_samples

        # Number of stride-sized chunks needed to cover original_length
        # (ceil division)
        num_strides = (original_length + stride - 1) // stride

        # Total length required so that:
        # - audio length is a multiple of stride
        # - plus lookahead tail for the final window
        required_length = num_strides * stride + lookahead_samples

        pad_length = max(0, required_length - original_length)

        if pad_length > 0:
            pad = src_audio.new_zeros((*src_audio.shape[:-1], pad_length))
            audio_padded = torch.cat([src_audio, pad], dim=-1)
        else:
            audio_padded = src_audio


        all_audios = []
        all_z_embeds = []
        with resynthesizer.encoder.net.streaming(batch_size=1):
            with resynthesizer.decoder.streaming(batch_size=1):
                for i, offset in tqdm(enumerate(range(0, original_length, stride))):  # in steps of frame_size, so chunk size is 20ms
                    frame = audio_padded[:, :, offset:offset + stride]
                    if model_name == "DarkStream-quant-TVT":
                        frame = audio_padded[:, :, offset:offset + stride + lookahead_samples] # add lookahead for quant model
    
                    # print("frame shape", frame.shape)
                    if use_projector:
                        z_embed, _ = resynthesizer.encoder.net(frame)
                    else:
                        z_embed = resynthesizer.encoder.net.encode_to_latent(frame, tr=lookahead_size)
                    if isinstance(z_embed, tuple):
                        z_embed = z_embed[0]
                    if z_embed.shape[-1] == 0:
                        continue
                    if use_projector:
                        z_embed = resynthesizer.projector(z_embed.transpose(1, 2)).transpose(1, 2)

                    if model_name == "DarkStream-quant-TVT":
                        z_embed = z_embed[:, :, :chunk_size]
                    resynth_audio_chunk, _ = resynthesizer.decoder(z_embed, tgt_embed)

                    all_audios.append(resynth_audio_chunk)
                    all_z_embeds.append(z_embed)
    all_audios = torch.cat(all_audios, dim=-1)
    all_z_embeds = torch.cat(all_z_embeds, dim=-1)

    resyth_audio = all_audios.squeeze().cpu().numpy()
    resyth_audio = resyth_audio * MAX_WAV_VALUE
    resyth_audio = resyth_audio.astype('int16')
    return resyth_audio

In [ ]:
def inference_latransformer(source_audio_path, tgt_embed, sample_rate=16000):
    src_audio, sr = librosa.load(source_audio_path, sr=sample_rate)
    assert sr == sample_rate
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    resynthesizer.eval()
    with torch.no_grad():
        original_length = src_audio.shape[-1]
        chunk_size = 3
        frame_samples = 320
        lookahead_size = 4

        # The model advances by 'stride' frames each step.
        stride = chunk_size * frame_samples

        # We always need lookahead frames available for the last step.
        lookahead_samples = lookahead_size * frame_samples

        # Number of stride-sized chunks needed to cover original_length
        # (ceil division)
        num_strides = (original_length + stride - 1) // stride

        # Total length required so that:
        # - audio length is a multiple of stride
        # - plus lookahead tail for the final window
        required_length = num_strides * stride + lookahead_samples

        pad_length = max(0, required_length - original_length)

        if pad_length > 0:
            pad = src_audio.new_zeros((*src_audio.shape[:-1], pad_length))
            audio_padded = torch.cat([src_audio, pad], dim=-1)
        else:
            audio_padded = src_audio


        all_audios = []
        all_z_embeds = []
        embed_stack = torch.empty((1, 512, 0)).to(device)
        embed_pointer = 0
        with resynthesizer.encoder.net.streaming(batch_size=1):
            with resynthesizer.decoder.streaming(batch_size=1):
                for offset in tqdm(range(0, original_length, stride + lookahead_samples)):  # in steps of frame_size, so chunk size is 20ms
                    frame = audio_padded[:, :, offset:offset + stride + lookahead_samples] # add lookahead for quant model
    
                    enc_state = resynthesizer.encoder.net._streaming_state
                    with resynthesizer.encoder.net._context_for_encoder_decoder:
                        emb = resynthesizer.encoder.net.encoder(frame)

                    embed_stack = torch.cat((embed_stack, emb), dim=-1)
                    if embed_stack.shape[-1] - embed_pointer < lookahead_size + chunk_size:
                        continue

                    while embed_stack.shape[-1] - embed_pointer >= lookahead_size + chunk_size:
                        emb = embed_stack[:, :, embed_pointer:embed_pointer + lookahead_size + chunk_size]
                        embed_pointer += chunk_size

                        assert enc_state.graphed_tr_enc is not None
                        (emb,) = enc_state.graphed_tr_enc(emb, lookahead_size)
                        z_embed, *_ = resynthesizer.encoder.net.quantizer(emb)
                        if isinstance(z_embed, tuple):
                            z_embed = z_embed[0]
                        if z_embed.shape[-1] == 0:
                            continue

                        z_embed = z_embed[:, :, :chunk_size]
                        resynth_audio_chunk, _ = resynthesizer.decoder(z_embed, tgt_embed)

                        all_audios.append(resynth_audio_chunk)
                        all_z_embeds.append(z_embed)
    all_audios = torch.cat(all_audios, dim=-1)
    all_z_embeds = torch.cat(all_z_embeds, dim=-1)

    resyth_audio = all_audios.squeeze().cpu().numpy()
    resyth_audio = resyth_audio * MAX_WAV_VALUE
    resyth_audio = resyth_audio.astype('int16')
    return resyth_audio

In [17]:
wav_file = "/data/waris/code/DarkStreamExt/sample_wav/genvc/EF4_ENG_0112_1.wav"
_device = "cuda"
sample_rate = 16000
audio, _ = librosa.load(wav_file, sr=sample_rate)
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))

resyth_audio = inference(wav_file, embed, model_name, sample_rate=16000, use_projector=False)

disp.Audio(resyth_audio, rate=16000)

26it [00:01, 24.63it/s]


In [16]:
wav_file = "/data/waris/code/DarkStreamExt/sample_wav/genvc/EF4_ENG_0112_1.wav"
_device = "cuda"
sample_rate = 16000
audio, _ = librosa.load(wav_file, sr=sample_rate)
audio = torch.from_numpy(audio).unsqueeze(0).unsqueeze(0).to(_device)
embed = speaker_encoder(audio.squeeze(0))

resyth_audio = inference_latransformer(wav_file, embed, sample_rate=16000)

disp.Audio(resyth_audio, rate=16000)

100%|██████████| 11/11 [00:01<00:00,  8.98it/s]
